In [ ]:
# Different graphs/scenarios from the original figure 2C, which is a single graph showing the effect of competition on the condensate landscape.
# A = chemical reactions of a single node
# B = chemical reactions represented as an artificial neural network (ANN) node
# C = effect of binding kinetics / ideal binding kinetics (low competition) 
# D = effect of (total) resource availability
# E = competing node (S2)
# F = antisigma (A2) / reduced competition (A2 binds S2)
# G = no competition 
# H = with competition

# Changing the code to fit a different data set:

    # 1. change how & what data is read in (i.e. alter inputs)
    # 2. modify inputs x1 and x2
    #3. make sure new dataset fits inputs

# New code huzzah!
# Note: the code is structured to be modular, so you can easily swap out the data reading and input generation sections without affecting the core simulation and plotting logic (just ensure that your new dataset has the necessary columns for scaling and mapping onto the heatmap grid)
import pandas as pd
import numpy as np
import scipy.integrate
from sklearn.preprocessing import MinMaxScaler
import bokeh.io
import bokeh.plotting
from bokeh.layouts import row
from bokeh.models import LinearColorMapper, ColumnDataSource, ColorBar

# Initialize Bokeh for Jupyter Notebook
bokeh.io.output_notebook()

# Read in data and preprocess it 
csv_filename = "figure2_panelc.csv"
x_column = "Cdil (mM)"
y_column = "DGtr (kcal/mol)"
tag_column = "Tag"

try:
    # converts "Tag" column to string and removes leading/trailing whitespace (slay)
    df = pd.read_csv(csv_filename)
    df[tag_column] = df[tag_column].astype(str).str.strip()
    
    # Scale x and y to [0, 1] for consistent mapping onto the heatmap grid
    scaler = MinMaxScaler(feature_range=(0, 1))
    df[['scaled_x', 'scaled_y']] = scaler.fit_transform(df[[x_column, y_column]])
    
    # Separate GFP and mCherry data based on the "Tag" column
    gfp_data = df[df[tag_column].str.contains('gfp', case=False, na=False)]
    mcherry_data = df[df[tag_column].str.contains('mcherry', case=False, na=False)]
    
    # Extract scaled coordinates for both tags
    gfp_x, gfp_y = gfp_data['scaled_x'].values, gfp_data['scaled_y'].values
    mcherry_x, mcherry_y = mcherry_data['scaled_x'].values, mcherry_data['scaled_y'].values

# Error handling for file not found   
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find '{csv_filename}'. Ensure it is in the same folder!")

# Single species sequestration model with non-competitive inhibition by a shared pool of free "C" molecules
# Basically has to be non competitive because the condensates aren't competing for resources, they're just sequestering them away from the soluble pool

# Kinetics of a 2-node competitive biomolecular model with a shared resource pool
# In phase separation: ct = Scaffold Capacity, C1 & C2 = Condensate Mass

# fundamental equation block of neural netowrk kinetics with competition for a shared resource pool
   # Si = sigma factors (activators)
   # Ai = antisigma factors (inhibitors)
   # C = shared resource pool (RNA polymerase) that signma factors compete to bind
   # Ci = active complexes or outputs of the nodes once they bind the shared resource

# ODE system 
def Sequestration_rhs(x, t, a1, b1, a2, b2, g1, g2, d, ct):
    S1, A1, S2, A2, C1, C2 = x
    C = ct - C1 - C2 
    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C,
        b1 - d*A1 - g1*A1*S1,
        0, 0,
        g2*S1*C - d*C1,
        0
    ])

# Parameter choices for the simulations (these can be varied to explore different regimes)
g1_choices = [10, 100, 1000.]
g2 = 10.0
d = 1.0
ct = 1/5

t = np.linspace(0, 5, 100)
x0 = np.array([0., 0., 0., 0., 0., 0.])

# Generate the heatmap data for each g1 choice and also compute the model output at the experimental data points for overlaying on the heatmap
N_grid = 21
grid_axis = np.linspace(0, 1, N_grid)
heatmap_matrix = np.zeros((N_grid, N_grid, 3))
scatter_gfp = np.zeros((len(gfp_x), 3))
scatter_mcherry = np.zeros((len(mcherry_x), 3))

# The simulations themselves
for k, g1k in enumerate(g1_choices):
    for i, x1i in enumerate(grid_axis):
        for j, x2j in enumerate(grid_axis):
            x_sim = scipy.integrate.odeint(Sequestration_rhs, x0, t, args=(x1i, x2j, 0, 0, g1k, g2, d, ct))
            heatmap_matrix[j, i, k] = x_sim.transpose()[4, -1] 

    for idx in range(len(gfp_x)):
        x_pt = scipy.integrate.odeint(Sequestration_rhs, x0, t, args=(gfp_x[idx], gfp_y[idx], 0, 0, g1k, g2, d, ct))
        scatter_gfp[idx, k] = x_pt.transpose()[4, -1]
        
    for idx in range(len(mcherry_x)):
        x_pt = scipy.integrate.odeint(Sequestration_rhs, x0, t, args=(mcherry_x[idx], mcherry_y[idx], 0, 0, g1k, g2, d, ct))
        scatter_mcherry[idx, k] = x_pt.transpose()[4, -1]

# Creating heatmap and scatter plot overlays
fig_size = (250, 250)
x_range = (grid_axis[0], grid_axis[-1])
y_range = (grid_axis[0], grid_axis[-1])

palette = bokeh.palettes.Viridis256
phase_mapper = LinearColorMapper(palette=palette, low=0, high=ct)

plots = []

for k, g1k in enumerate(g1_choices):
    p = bokeh.plotting.figure(
        title=f"NPM1 Landscape (g1={g1k})", width=fig_size[0], height=fig_size[1],
        x_range=x_range, y_range=y_range,
        x_axis_label="Norm. Cdil", y_axis_label="Norm. DGtr" if k==0 else ""
    )
    
    # Adjusting borders to prevent the color bar from being cut off and to give more space for the legend
    p.min_border_left = 50
    p.min_border_right = 20
    p.min_border_top = 30
    p.min_border_bottom = 50
    
    z0 = np.c_[heatmap_matrix[:, :, k], np.zeros(N_grid)]
    z0[-1, -1] = ct  
    p.image(image=[z0], x=grid_axis.min(), y=grid_axis.min(), 
            dw=(grid_axis.max() - grid_axis.min()) * (1 + 1/N_grid), 
            dh=grid_axis.max() - grid_axis.min(), palette=palette, alpha=0.85)
    
    if len(gfp_x) > 0:
        source_gfp = ColumnDataSource(data=dict(x=gfp_x, y=gfp_y, out=scatter_gfp[:, k]))
        p.scatter('x', 'y', source=source_gfp, size=8, 
                  fill_color="#2ECC71", line_color="#1E8449", line_width=1.0, 
                  legend_label="GFP-NPM1")
        
    if len(mcherry_x) > 0:
        source_mch = ColumnDataSource(data=dict(x=mcherry_x, y=mcherry_y, out=scatter_mcherry[:, k]))
        p.scatter('x', 'y', source=source_mch, size=8, 
                  fill_color="#E74C3C", line_color="#922B21", line_width=1.0, 
                  legend_label="mCh-NPM1")
        
    p.legend.location = "bottom_right"
    p.legend.label_text_font_size = "7pt"
    plots.append(p)

# Create a separate plot for the color bar to ensure it doesn't get cut off and is properly aligned with the heatmaps
colorbar_plot = bokeh.plotting.figure(
    width=90, height=fig_size[1], 
    toolbar_location=None, 
    min_border=0
)

# Dummy invisible point to satisfy Bokeh's validation rules
colorbar_plot.scatter(x=[0], y=[0], alpha=0)

# Hide axes, grid, and borders completely
colorbar_plot.xaxis.visible = False
colorbar_plot.yaxis.visible = False
colorbar_plot.grid.grid_line_color = None
colorbar_plot.outline_line_color = None

# Configure the ColorBar
colorbar = ColorBar(
    color_mapper=phase_mapper, 
    location=(0, 0), 
    width=15, 
    title="Condensate Density"
)
colorbar_plot.add_layout(colorbar, 'left')

# Combine the three identical-sized graphs and the standalone colorbar
final_layout = row(*plots, colorbar_plot)

# Final graph huzzah!
bokeh.io.show(final_layout)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Generating synthetic experimental data
np.random.seed(42)
N_samples = 400

# Generating a simulated experimental feature space
X = np.random.uniform(-3, 3, size=(N_samples, 2))

# True underlying experimental relationship (y = 2*x0^2 - 3*x1 + noise)
y_true = 2 * (X[:, 0]**2) - 3 * X[:, 1]
noise = np.random.normal(0, 1.5, size=N_samples)
y_experimental = y_true + noise

# Set up 5-fold cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Track performance evaluation metrics across folds
train_errors = []
test_errors = []

# Array to store predictions for the entire dataset
predictions = np.zeros(N_samples)

# Initialize the machine learning model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Running the 5-Fold Evaluation Loop
print("Starting 5-Fold Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    # Split the dataset into training and validation sets for this fold
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_experimental[train_idx], y_experimental[test_idx]
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict outcomes
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Save the test predictions into the corresponding indices
    predictions[test_idx] = y_test_pred
    
    # Calculate Mean Squared Error (MSE)
    mse_train = mean_squared_error(y_train, y_train_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)
    
    train_errors.append(mse_train)
    test_errors.append(mse_test)
    
    print(f"Fold {fold + 1} -> Train MSE: {mse_train:.3f} | Test MSE: {mse_test:.3f}")

# Calculate final aggregated summary stats
avg_test_mse = np.mean(test_errors)
std_test_mse = np.std(test_errors)

print("\n==================================================")
print(f"Final Performance Summary:")
print(f"Average Overall Test MSE: {avg_test_mse:.3f} (± {std_test_mse:.3f})")
print("==================================================\n")

# Plot Predicted vs Experimental values with the average test MSE in the title
plt.figure(figsize=(7, 7))

# Scatter plot of all experimental values vs cross-validated predictions (Light Sage Green)
plt.scatter(y_experimental, predictions, alpha=0.7, color='#a1dbcd', edgecolor='#556b2f', linewidth=0.8, label='Predicted Data')

# Draw perfect prediction identity reference line (y = x)
min_val = min(y_experimental.min(), predictions.min())
max_val = max(y_experimental.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color='tomato', linestyle='--', lw=2, label='Perfect Fit')

# Include the average validation metric inside the plot title dynamically
plt.title(f"Predicted vs. Experimental (Avg Test MSE: {avg_test_mse:.2f})", fontsize=13, fontweight='bold')
plt.xlabel("Experimental (True Values)", fontsize=11)
plt.ylabel("Predicted Values", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=10, loc='lower right')
plt.axis('equal')  # Forces a square aspect ratio so the 45-degree line looks accurate

plt.tight_layout()
plt.show()

In [1]:
import numpy as np
import scipy.integrate

import bokeh.io
import bokeh.plotting

from bokeh.layouts import gridplot, row, column
from bokeh.models import LinearColorMapper, ColorBar

# Initialize Bokeh for Jupyter Notebook
bokeh.io.output_notebook()

# Non-competitive model with a single node and a shared resource pool (C) that sequesters the activator (S1) without direct competition from another node
def NonCompetitive_rhs(x, t, a1, b1, g1, g2, d, ct):

    S1, A1, C1 = x

    C = ct - C1

    return np.array([
        a1 - d*S1 - g1*A1*S1 - g2*S1*C,
        b1 - d*A1 - g1*A1*S1,
        g2*S1*C - d*C1
    ])

# Competitive extension of the model with a second node (S2, A2) that also sequesters from the shared resource pool (C), creating indirect competition between the two nodes for the shared resource
# Purely hypothetical competitor node with its own production and degradation kinetics, but it also sequesters from the same shared resource pool (C) that the original node (S1, A1) is trying to access, thus creating competition for the shared resource
def Competitive_rhs(x, t, a1, b1, a2, b2, g1, g2, d, ct):

    S1, A1, S2, A2, C1, C2 = x

    C = ct - C1 - C2

    return np.array([
        # NPM1 species
        a1 - d*S1 - g1*A1*S1 - g2*S1*C,
        b1 - d*A1 - g1*A1*S1,

        # Competitor species
        a2 - d*S2 - g1*A2*S2 - g2*S2*C,
        b2 - d*A2 - g1*A2*S2,

        # Condensates
        g2*S1*C - d*C1,
        g2*S2*C - d*C2
    ])

# Parameters 
g1_choices = [10, 100, 1000]

g2 = 10.0
d = 1.0
ct = 0.2

# Hypothetical competitor
competitor_a = 0.30
competitor_b = 0.30

t = np.linspace(0, 5, 100)

# Grid
N_grid = 41
grid_axis = np.linspace(0, 1, N_grid)

heatmap_noncomp = np.zeros((N_grid, N_grid, 3))
heatmap_comp = np.zeros((N_grid, N_grid, 3))

# Generate landscapes
for k, g1k in enumerate(g1_choices):

    for i, x1i in enumerate(grid_axis):
        for j, x2j in enumerate(grid_axis):

            # Original noncompetitive model
            x_sim = scipy.integrate.odeint(
                NonCompetitive_rhs,
                np.array([0., 0., 0.]),
                t,
                args=(
                    x1i,   # production/input
                    x2j,   # inhibitor/barrier
                    g1k,
                    g2,
                    d,
                    ct
                )
            )

            heatmap_noncomp[j, i, k] = x_sim[-1, 2]

            # Hypothetical competitive extension model
            x_sim = scipy.integrate.odeint(
                Competitive_rhs,
                np.array([0., 0., 0., 0., 0., 0.]),
                t,
                args=(
                    x1i,
                    x2j,
                    competitor_a,
                    competitor_b,
                    g1k,
                    g2,
                    d,
                    ct
                )
            )

            heatmap_comp[j, i, k] = x_sim[-1, 4]

# Plot landscapes together
palette = bokeh.palettes.Viridis256

phase_mapper = LinearColorMapper(
    palette=palette,
    low=0,
    high=ct
)

plot_rows = []
fig_size = (300, 300)

for model_type in ["noncomp", "comp"]:

    current_row = []

    for k, g1k in enumerate(g1_choices):

        title = (
            f"Noncompetitive (g1={g1k})"
            if model_type == "noncomp"
            else
            f"Competitive (g1={g1k})"
        )

        p = bokeh.plotting.figure(
            title=title,
            width=fig_size[0],
            height=fig_size[1],
            x_range=(0, 1),
            y_range=(0, 1),
            x_axis_label="Normalized Cdil",
            y_axis_label="Normalized DGtr"
        )

        # Explicitly enforce borders so titles and axis ticks don't break grid sizing symmetry
        p.min_border_left = 60
        p.min_border_right = 20
        p.min_border_top = 40
        p.min_border_bottom = 50

        if model_type == "noncomp":
            z = heatmap_noncomp[:, :, k]
        else:
            z = heatmap_comp[:, :, k]

        z0 = np.c_[z, np.zeros(N_grid)]
        z0[-1, -1] = ct

        p.image(
            image=[z0],
            x=0,
            y=0,
            dw=1 + 1/N_grid,
            dh=1,
            palette=palette
        )

        current_row.append(p)

    plot_rows.append(current_row)

# Standalone Color Bar Plot Assembly
colorbar_plot = bokeh.plotting.figure(
    width=90, 
    height=fig_size[1] * 2, 
    toolbar_location=None, 
    min_border=0
)

# Invisible scatter node renderer to intentionally satisfy W-1000 validation checks
colorbar_plot.scatter(x=[0], y=[0], alpha=0)

# Hide coordinate layers completely
colorbar_plot.xaxis.visible = False
colorbar_plot.yaxis.visible = False
colorbar_plot.grid.grid_line_color = None
colorbar_plot.outline_line_color = None

colorbar = ColorBar(
    color_mapper=phase_mapper,
    width=15,
    location=(0, 0),
    title="Condensate Density"
)
colorbar_plot.add_layout(colorbar, 'left')

# Display
heatmap_grid = gridplot(plot_rows)
final_layout = row(heatmap_grid, colorbar_plot)

bokeh.io.show(final_layout)

<class 'ModuleNotFoundError'>: No module named 'bokeh'